# 03. Label Data with ChatGPT/LLM

This notebook handles the classification of job descriptions using the OpenAI API.

In [ ]:
# --- IMPORTS ---
import pandas as pd
from openai import OpenAI
import time
import os
from tqdm.notebook import tqdm
from dotenv import load_dotenv

In [ ]:
# --- CONFIGURATION ---
load_dotenv()  # Load environment variables from .env

API_KEY = os.getenv("OPENAI_API_KEY")
if not API_KEY:
    raise ValueError("OPENAI_API_KEY not found in environment variables. Please check your .env file.")

# Use standard paths or configurable ones
INPUT_FILE = 'data/final_3000_balanced_longtext.csv'
OUTPUT_FILE = 'data/dataset_STACKOVERFLOW.csv' # Or 'data/labeled_output.csv' if you prefer to organize it

# Start index configuration
START_FROM_ROW = 3000

categories = [
    'web frontend', 'web backend', 'mobile', 'data_ai', 'devops_cloud',
    'testing_qa', 'security', 'embedded', 'iot', 'game',
    'blockchain', 'enterprise', 'developer tools'
]

In [ ]:
# --- FUNCTIONS ---

def classify_job(client, text):
    prompt = f"""
    Act as an expert IT Recruiter.
    Analyze the job description below and classify it into EXACTLY ONE of these categories:
    {categories}

    Constraints:
    1. Return ONLY the category name (lowercase).
    2. Do NOT add explanations or punctuation.
    3. If it fits multiple, choose the most dominant one.

    Job Description:
    '''
    {text}
    '''
    """
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "You are a precise data classification assistant."},
                {"role": "user", "content": prompt}
            ],
            temperature=0,
            max_tokens=20
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f" -> API Error: {e}")
        time.sleep(2)
        return "error"

In [ ]:
# --- MAIN EXECUTION ---

client = OpenAI(api_key=API_KEY)

# 1. RESUME Logic: Check for existing output file
if os.path.exists(OUTPUT_FILE):
    print(f"Found '{OUTPUT_FILE}'. Loading existing data to resume...")
    df = pd.read_csv(OUTPUT_FILE)
else:
    print(f"Starting fresh from '{INPUT_FILE}'...")
    try:
        df = pd.read_csv(INPUT_FILE)
    except FileNotFoundError:
        print(f"ERROR: Input file {INPUT_FILE} not found.")
        # In a notebook, we might want to stop execution here, but raising an error is also fine
        raise FileNotFoundError(f"Input file {INPUT_FILE} not found.")

# Create label_gpt column if it doesn't exist
if 'label_gpt' not in df.columns:
    df['label_gpt'] = ""

# Normalize label_gpt column to string
df['label_gpt'] = df['label_gpt'].astype(str)

total_rows = len(df)

print(f"\nTotal rows in file: {total_rows}")
print(f"Processing starting from index: {START_FROM_ROW} (Row number {START_FROM_ROW + 1})")
print("-" * 60)

# 2. Processing Loop
for index, row in tqdm(df.iterrows(), total=total_rows):

    # === CONDITION: SKIP ROWS BEFORE 3000 ===
    if index < START_FROM_ROW:
        continue
    # ========================================

    current_label = row['label_gpt']

    # Run only on rows without a label
    if current_label == "nan" or current_label.strip() == "" or current_label == "error":

        raw_text = str(row['raw_text'])

        # Call API
        result = classify_job(client, raw_text)

        # Save to dataframe
        df.at[index, 'label_gpt'] = result

        # Optional: Print progress less frequently in notebook to avoid clutter
        # tqdm.write(f"{index:<5} | {'DONE':<10} | {result}")

        # Save file every 50 rows
        if index % 50 == 0:
            df.to_csv(OUTPUT_FILE, index=False)

    else:
        # Skip if already labeled
        pass

# 3. Final Save
df.to_csv(OUTPUT_FILE, index=False)
print("\n" + "=" * 50)
print(f"COMPLETED! Result saved to: {OUTPUT_FILE}")